# Notebook 1: Markov Processes
### Week 2 — David Silver's RL Course

In this notebook we explore **Markov Processes** (also called Markov Chains): the foundation of everything that follows in this course.

**Contents:**
1. The Markov Property
2. State Transition Matrices
3. Simulating the Student Markov Chain
4. Visualising transition dynamics
5. Exercises

## 1. The Markov Property

A state $S_t$ is **Markov** if and only if:

$$P[S_{t+1} \mid S_t] = P[S_{t+1} \mid S_1, S_2, \ldots, S_t]$$

In plain English: **knowing the current state is sufficient to predict the future** — the history can be thrown away.

This is a powerful assumption. It lets us represent the entire dynamics of an environment with a single matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

np.random.seed(42)
print("Libraries loaded. Let's build some Markov Chains!")

## 2. The Student Markov Chain

We'll use the running example from the lecture. A student moves between these states:

| State | Description |
|-------|-------------|
| C1    | Class 1     |
| C2    | Class 2     |
| C3    | Class 3     |
| FB    | Facebook    |
| Pub   | Pub         |
| Pass  | Pass exam   |
| Sleep | Sleep (terminal) |

The state **Sleep** is *absorbing* — once you get there, you stay.

In [ ]:
# ── State definitions ──────────────────────────────────────────────────────
states = ['C1', 'C2', 'C3', 'Pass', 'Pub', 'FB', 'Sleep']
n = len(states)
idx = {s: i for i, s in enumerate(states)}   # name → index lookup

# ── Transition matrix (rows = from, cols = to) ─────────────────────────────
# Each row must sum to 1.
P = np.zeros((n, n))

# From C1: 50% → C2, 50% → FB
P[idx['C1'], idx['C2']] = 0.5
P[idx['C1'], idx['FB']] = 0.5

# From C2: 80% → C3, 20% → Sleep
P[idx['C2'], idx['C3']] = 0.8
P[idx['C2'], idx['Sleep']] = 0.2

# From C3: 60% → Pass, 40% → Pub
P[idx['C3'], idx['Pass']] = 0.6
P[idx['C3'], idx['Pub']] = 0.4

# From Pass: 100% → Sleep  (terminal state)
P[idx['Pass'], idx['Sleep']] = 1.0

# From Pub: 20% → C1, 40% → C2, 40% → C3
P[idx['Pub'], idx['C1']] = 0.2
P[idx['Pub'], idx['C2']] = 0.4
P[idx['Pub'], idx['C3']] = 0.4

# From FB: 90% → FB (stay), 10% → C1
P[idx['FB'], idx['FB']] = 0.9
P[idx['FB'], idx['C1']] = 0.1

# From Sleep: stays (absorbing)
P[idx['Sleep'], idx['Sleep']] = 1.0

# ── Sanity check ────────────────────────────────────────────────────────────
assert np.allclose(P.sum(axis=1), 1.0), "Row sums must equal 1!"
print("Transition matrix verified — all rows sum to 1.\n")

# Pretty-print the matrix
import pandas as pd
df = pd.DataFrame(P, index=states, columns=states)
print("Transition Matrix P:")
print(df.to_string(float_format=lambda x: f"{x:.1f}" if x > 0 else " . "))

## 3. Sampling Episodes

An **episode** is a sequence of states $S_1, S_2, \ldots, S_T$ sampled from the chain. We start from $S_1 = C1$ and run until we reach the absorbing state Sleep.

In [ ]:
def sample_episode(P, states, idx, start='C1', max_steps=50):
    """Sample one episode from the Markov chain."""
    episode = [start]
    current = start
    for _ in range(max_steps):
        if current == 'Sleep':
            break
        next_state = np.random.choice(states, p=P[idx[current]])
        episode.append(next_state)
        current = next_state
    return episode


# ── Sample 5 episodes (matching the lecture slides) ─────────────────────────
np.random.seed(0)
print("Sample episodes starting from C1:\n")
for i in range(5):
    ep = sample_episode(P, states, idx, start='C1')
    print(f"Episode {i+1}: {' → '.join(ep)}")

## 4. Visualising the Transition Matrix

A heatmap lets us immediately see which transitions are possible and how probable they are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Heatmap ─────────────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(P, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(states)
ax.set_yticks(range(n)); ax.set_yticklabels(states)
ax.set_xlabel('Next State'); ax.set_ylabel('Current State')
ax.set_title('Transition Matrix Heatmap', fontsize=13, fontweight='bold')
for i in range(n):
    for j in range(n):
        if P[i, j] > 0:
            ax.text(j, i, f'{P[i,j]:.1f}', ha='center', va='center',
                    fontsize=9, color='white' if P[i,j] > 0.6 else 'black')
plt.colorbar(im, ax=ax, label='Transition Probability')

# ── Episode length distribution ─────────────────────────────────────────────
ax2 = axes[1]
np.random.seed(1)
lengths = [len(sample_episode(P, states, idx)) for _ in range(2000)]
ax2.hist(lengths, bins=range(1, max(lengths)+2), color='steelblue',
         edgecolor='white', alpha=0.85)
ax2.set_xlabel('Episode Length (steps)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Episode Lengths (n=2000)', fontsize=13, fontweight='bold')
ax2.axvline(np.mean(lengths), color='red', linestyle='--',
            label=f'Mean = {np.mean(lengths):.1f}')
ax2.legend()

plt.tight_layout()
plt.savefig('markov_chain_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Average episode length: {np.mean(lengths):.2f} steps")

## 5. Multi-Step Transition Probabilities

The $n$-step transition probability $P^n$ tells us: starting from state $s$, what is the probability of being in state $s'$ after exactly $n$ steps?

$$P^n = \underbrace{P \cdot P \cdots P}_{n \text{ times}}$$

In [ ]:
def n_step_transition(P, n):
    """Compute the n-step transition matrix."""    result = np.eye(len(P))
    for _ in range(n):
        result = result @ P
    return result

start = idx['C1']
print(f"Starting from C1, probability of being in each state after k steps:\n")
print(f"{'State':<8}", end='')
for k in [1, 2, 3, 5, 10, 20]:
    print(f"  k={k:<4}", end='')
print()
print("-" * 55)

for s in states:
    print(f"{s:<8}", end='')
    for k in [1, 2, 3, 5, 10, 20]:
        Pk = n_step_transition(P, k)
        print(f"  {Pk[start, idx[s]]:.4f}", end='')
    print()

print("\nNotice: probability of Sleep approaches 1.0 as k increases.")

## 6. Stationary Distribution (for non-absorbing chains)

For a chain *without* an absorbing state, the **stationary distribution** $d$ satisfies:
$$d^\top P = d^\top$$

This means once the chain reaches stationarity, the probability of being in each state no longer changes with time.

We find it as the left eigenvector of $P$ corresponding to eigenvalue 1.

In [ ]:
# Build a cyclic chain (no absorbing state) to demonstrate stationarity
# Use only {C1, C2, C3, FB, Pub} with modified transitions

states_c = ['C1', 'C2', 'C3', 'FB', 'Pub']
nc = len(states_c)
idxc = {s: i for i, s in enumerate(states_c)}

Pc = np.zeros((nc, nc))
Pc[idxc['C1'], idxc['C2']] = 0.5;  Pc[idxc['C1'], idxc['FB']]  = 0.5
Pc[idxc['C2'], idxc['C3']] = 0.8;  Pc[idxc['C2'], idxc['C1']]  = 0.2
Pc[idxc['C3'], idxc['Pub']]= 1.0
Pc[idxc['FB'], idxc['FB']] = 0.9;  Pc[idxc['FB'], idxc['C1']]  = 0.1
Pc[idxc['Pub'],idxc['C1']] = 0.2;  Pc[idxc['Pub'],idxc['C2']] = 0.4
Pc[idxc['Pub'],idxc['C3']] = 0.4
assert np.allclose(Pc.sum(axis=1), 1.0)

# Stationary distribution via left eigenvector (eigenvalue = 1)
eigvals, eigvecs = np.linalg.eig(Pc.T)
stationary = eigvecs[:, np.argmax(np.real(eigvals))]
stationary = np.real(stationary)
stationary /= stationary.sum()

print("Stationary distribution of the cyclic chain:")
for s, p in zip(states_c, stationary):
    bar = '█' * int(p * 40)
    print(f"  {s:<6} {p:.4f}  {bar}")

# Verify: d @ P ≈ d
assert np.allclose(stationary @ Pc, stationary, atol=1e-8)
print("\nVerification: d @ P == d ✓")

## 7. Exercises

Try these yourself to deepen your understanding.

**Exercise 1:** Modify the transition matrix so that from `C3`, the student goes to Pub with probability 0.8 and Pass with probability 0.2. Re-run the episode sampler. What happens to average episode length?

**Exercise 2:** Starting from `FB` (Facebook), compute the probability of eventually reaching `Pass`. (*Hint: use the n-step matrix for large n.*)

**Exercise 3:** Write a function `visit_counts(P, states, idx, start, n_episodes)` that returns the fraction of time spent in each state across many episodes. Compare to the stationary distribution.

**Exercise 4 (challenge):** Implement the **power iteration** method to find the stationary distribution instead of using `np.linalg.eig`.

In [ ]:
# ── Exercise 1: your code here ──────────────────────────────────────────────


# ── Exercise 2: your code here ──────────────────────────────────────────────


# ── Exercise 3: your code here ──────────────────────────────────────────────


# ── Exercise 4: power iteration ─────────────────────────────────────────────
def power_iteration_stationary(P, tol=1e-9, max_iter=10000):
    """Find stationary distribution via power iteration.
    Start with a uniform distribution and repeatedly apply P."""
    d = np.ones(len(P)) / len(P)   # uniform start
    # TODO: iterate d = d @ P until convergence
    pass

# d = power_iteration_stationary(Pc)
# print(d)